## Imports

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../src/data/pokedex.csv')
display(df.columns)
display(df)

## Feature engineering

In [ ]:
# Pour le modèle, on veut l'id du Pokémon et son/ses type(s) (variable cible)
# On récupère également le poids, la taille, l'expérience de base et le bonheur de base
df_model = df[["id", "type_1", "type_2", "height", "weight", "base_experience", "base_happiness"]].copy()


# On a besoin du total des statistiques de combat, ainsi que la répartition de ces statistiques
df_model["total_stats"] = df["hp"] + df["attack"] + df["defense"] + df["special-attack"] + df["special-defense"] + df["speed"]

stats = ["hp", "attack", "defense", "special-attack", "special-defense", "speed"]
for stat in stats:
	df_model[f"{stat}_pct"] = df[stat] * 100 / df_model["total_stats"]


# Le Pokémon a peut-être des statistiques supérieures parce qu'il est spécial (légendaire, mythique, Ultra-Chimère, Paradoxe)
chimeras = [793, 794, 795, 796, 797, 798, 799, 803, 804, 805, 806]
paradoxs = [984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 1005, 1006, 1007, 1008, 1009, 1010, 1021, 1022, 1023]
df_model["is_special"] = (df["is_legendary"] | df["is_mythical"] | df_model["id"].isin(paradoxs) | df_model["id"].isin(chimeras))

# Pour le moment, on ne tient pas compte de capture_rate, gender_rate, growth_rate et hatch_counter

display(df_model.sort_values("id"))

In [ ]:
# Le total des statistiques peut aussi varier selon le niveau d'évolution du Pokémon
df_evolutions = df[["id", "name_en", "evolution_level", "evolves_from"]].copy()
babies = df.loc[df["is_baby"], "name_en"]

# Les Pokémons bébés ont un niveau d'évolution à -1
df_evolutions.loc[df_evolutions["name_en"].isin(babies), "evolution_level"] = -1

# Les Pokémons de base qui ont une pré-évolution bébé (comme Pikachu) ont un niveau d'évolution à 0
df_evolutions.loc[(df_evolutions["evolution_level"] == 1) & (df_evolutions["evolves_from"].isin(babies)), "evolution_level"] = 0
df_evolutions.loc[(df_evolutions["evolution_level"] == 0) & (df_evolutions["evolves_from"]), "evolves_from"] = np.nan

# Les Pokémons évoluant d'un Pokémon lui-même évolué (non bébé) ont un niveau d'évolution à 2
pokemons_lvl1 = df_evolutions.loc[df_evolutions["evolution_level"] == 1, "name_en"]
df_evolutions.loc[(df_evolutions["evolution_level"] == 1) & (df_evolutions["evolves_from"].isin(pokemons_lvl1)), "evolution_level"] = 2


df_model["evolution_level"] = df_evolutions["evolution_level"].copy()
df_model = df_model.sort_values("id")

display(df_model)